In [1]:
# =====================================================================
# Cell 1: Data Ingestion & Clinical Imputation (Robust Version)
# =====================================================================
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

heart_data = fetch_openml('heart-h', version=1, as_frame=True, parser='auto')

X = heart_data.data.copy()
y = heart_data.target.copy()

print(f"Raw features shape: {X.shape}")
print(f"Raw target shape: {y.shape}")

# --- ENGINEERING PIVOT: Robust Imputation instead of dropping ---
for col in X.columns:
    if X[col].dtype.name == 'category' or X[col].dtype == object:
        # Fill missing categorical features with the most frequent value (mode)
        most_frequent = X[col].mode()[0]
        X[col] = X[col].fillna(most_frequent)
    else:
        # Fill missing numerical features with the median
        median_value = X[col].median()
        X[col] = X[col].fillna(median_value)

# Explicitly map the categorical target string labels to clean binary integers
label_mapping = {
    '0': 0,
    '<50': 0,
    '>50_1': 1,
    '1': 1
}
y_clean = y.map(label_mapping).astype(int)

# Double-check: If any target values failed to map, drop just those rows
clean_mask = y_clean.notna()
X_clean = X[clean_mask]
y_clean = y_clean[clean_mask]

print(f"Cleaned features matrix shape: {X_clean.shape}")
print(f"Cleaned target vector shape: {y_clean.shape}")
print(f"\nTarget Class Distribution:\n{y_clean.value_counts()}")

Raw features shape: (294, 13)
Raw target shape: (294,)
Cleaned features matrix shape: (294, 13)
Cleaned target vector shape: (294,)

Target Class Distribution:
num
0    188
1    106
Name: count, dtype: int64


In [2]:
# =====================================================================
# Cell 2: Train-Test Split & Baseline Model
# =====================================================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Explicit 80/20 train-test split with target stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.20, random_state=42, stratify=y_clean
)

print(f"Train set shape: {X_train.shape} | Test set shape: {X_test.shape}")

# Create the Random Forest model instance
rf = RandomForestClassifier(random_state=42)

try:
    # Attempt training on the processed features
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_test)
    
    # Evaluation outputs
    print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

except ValueError as e:
    print(f"Expected Error Caught: {e}")

Train set shape: (235, 13) | Test set shape: (59, 13)
Expected Error Caught: could not convert string to float: 'male'


In [3]:
# =====================================================================
# Cell 3: Feature Engineering & Preprocessing Pipeline
# =====================================================================
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Identify numerical and categorical columns automatically from X_clean
numerical_cols = X_clean.select_dtypes(include=['int64', 'float64', 'number']).columns.tolist()
categorical_cols = X_clean.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical features to scale: {numerical_cols}")
print(f"Categorical features to encode: {categorical_cols}\n")

# Define individual transformations for each data path
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Bundle both pathways into a single structural preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Combine the preprocessor and the model instance into a unified Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf)
])

# Fit the entire end-to-end pipeline on our stratified training sets
print("Training the preprocessed Random Forest pipeline...")
pipeline.fit(X_train, y_train)

# Evaluate the pipeline on the test set using your imported metrics
y_pred = pipeline.predict(X_test)

# Clean evaluation outputs
print(f"\n✅ Pipeline Trained Successfully!")
print(f"Preprocessed Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Numerical features to scale: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
Categorical features to encode: ['sex', 'chest_pain', 'fbs', 'restecg', 'exang', 'slope', 'thal']

Training the preprocessed Random Forest pipeline...

✅ Pipeline Trained Successfully!
Preprocessed Model Accuracy: 86.44%

              precision    recall  f1-score   support

           0       0.89      0.89      0.89        38
           1       0.81      0.81      0.81        21

    accuracy                           0.86        59
   macro avg       0.85      0.85      0.85        59
weighted avg       0.86      0.86      0.86        59

Confusion Matrix:
[[34  4]
 [ 4 17]]


In [4]:
# =====================================================================
# Cell 4: Pipeline Feature Importance Analysis
# =====================================================================

# 1. Reach into the pipeline preprocessor and pull out the one-hot text encoder names
cat_encoder = pipeline.named_steps['preprocessor'].named_transformers_['cat']
encoded_cat_cols = cat_encoder.get_feature_names_out(categorical_cols).tolist()

# 2. Combine all column names together into a single master list
all_features = numerical_cols + encoded_cat_cols

# 3. Pull out the actual mathematical weights from the final random forest model step
importances = pipeline.named_steps['model'].feature_importances_

# 4. Put them into a table and sort them from highest to lowest
importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

print(importance_df)

                          Feature  Importance
0                         oldpeak    0.157594
1                            chol    0.116065
2                         thalach    0.112913
3               chest_pain_asympt    0.102853
4                        trestbps    0.079713
5                       exang_yes    0.077483
6                             age    0.075626
7                        exang_no    0.053013
8          chest_pain_atyp_angina    0.047255
9                        sex_male    0.027408
10                     sex_female    0.025674
11         chest_pain_non_anginal    0.019432
12                       slope_up    0.013141
13              thal_fixed_defect    0.012162
14  restecg_st_t_wave_abnormality    0.011833
15                    thal_normal    0.011460
16                          fbs_t    0.010759
17         thal_reversable_defect    0.009752
18                          fbs_f    0.009257
19                 restecg_normal    0.008871
20                     slope_flat 